<a href="https://colab.research.google.com/github/Suchanekf/PROMPT-ENGINEERING-FOR-ENVIRONMENTAL-MODELING/blob/main/AI_Montclair_Showcase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Write Python code to train machine learning models to predict
“Daily Max 8-hour Ozone Concentration (ppm)” using the dataset
Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv.

Use appropriate preprocessing if necessary.
Split the data into training and testing sets.
Train RandomForestRegressor and GradientBoostingRegressor.
Report RMSE, MAE, and R² on the test set.

In [ ]:
Example structure:

1. Load dataset
2. Clean numeric columns
3. Define X and y
4. Perform train-test split
5. Train models
6. Evaluate using RMSE, MAE, R²

Now apply this exact workflow to
Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv
to predict “Daily Max 8-hour Ozone Concentration (ppm)”
using RandomForestRegressor and GradientBoostingRegressor.

In [ ]:
You are an environmental data scientist specializing in air quality modeling.

Write production-quality Python code to build a regression model that predicts
“Daily Max 8-hour Ozone Concentration (ppm)” from
Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv.

Ensure proper data cleaning, relevant feature engineering,
and robust model evaluation using RMSE, MAE, and R².

Train both RandomForestRegressor and GradientBoostingRegressor.

In [ ]:
Generate Python code and explicitly structure it in labeled steps:

Step 1: Load dataset
Step 2: Clean and preprocess data
Step 3: Engineer relevant features
Step 4: Define X and y
Step 5: Perform train-test split
Step 6: Train RandomForestRegressor and GradientBoostingRegressor
Step 7: Evaluate using RMSE, MAE, and R²

Dataset: Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv
Target: “Daily Max 8-hour Ozone Concentration (ppm)”

In [ ]:
Use an 80/20 train-test split with random_state=42.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv to Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


def to_num(s: pd.Series) -> pd.Series:
    x = s.astype(str).str.strip()
    x = x.replace({"": np.nan, "None": np.nan, "nan": np.nan, "NaN": np.nan})
    x = x.str.replace(",", "", regex=False)
    return pd.to_numeric(x, errors="coerce")


def main():
    file_path = "Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv"
    df = pd.read_csv(file_path)

    df.columns = df.columns.astype(str).str.strip()

    target_col = "Daily Max 8-hour Ozone Concentration (ppm)"
    precip_col = "Daily Precipitation (mm)"

    if target_col not in df.columns:
        raise ValueError(
            f"Target column not found: {target_col}. Available columns: {list(df.columns)}"
        )

    if precip_col in df.columns:
        df[precip_col] = (
            df[precip_col]
            .astype(str)
            .str.strip()
            .replace({"T": "0", "t": "0", "Trace": "0", "trace": "0"})
        )

    # Convert target
    df[target_col] = to_num(df[target_col])

    # Build numeric features:
    # - Start from all columns except target
    # - Convert to numeric
    # - Drop columns that are mostly NaN after conversion
    candidate_cols = [c for c in df.columns if c != target_col]

    # Drop obviously non-feature columns if present (safe no-op if absent)
    drop_like = ["date", "time", "station", "site", "county", "city", "state", "name", "id"]
    nonfeature = []
    for c in candidate_cols:
        cl = c.lower()
        if any(k in cl for k in drop_like):
            nonfeature.append(c)
    candidate_cols = [c for c in candidate_cols if c not in nonfeature]

    X_num = pd.DataFrame(index=df.index)
    for c in candidate_cols:
        if df[c].dtype.kind in "biufc":
            X_num[c] = df[c]
        else:
            X_num[c] = to_num(df[c])

    # Drop columns with too many missing values (keep columns with >= 30% non-missing)
    min_non_missing = int(np.ceil(0.30 * len(X_num)))
    X_num = X_num.dropna(axis=1, thresh=min_non_missing)

    if X_num.shape[1] == 0:
        raise ValueError("After numeric conversion, no feature columns remain (all were mostly missing or non-numeric).")

    # Keep rows where y exists and at least 1 feature exists (then fill remaining feature NaNs with median)
    data = pd.concat([X_num, df[target_col]], axis=1)
    data = data.dropna(subset=[target_col])

    X = data.drop(columns=[target_col])
    y = data[target_col]

    # Median impute (simple + stable for tree models)
    X = X.apply(lambda col: col.fillna(col.median()) if col.notna().any() else col.fillna(0))

    # If still any non-numeric sneak in, kill them
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(0)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )

    models = [
        ("RandomForestRegressor", RandomForestRegressor(random_state=42)),
        ("GradientBoostingRegressor", GradientBoostingRegressor(random_state=42)),
    ]

    results = []
    for model_name, model in models:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
        mae = float(mean_absolute_error(y_test, preds))
        r2 = float(r2_score(y_test, preds))

        results.append({
            "prompt_type": "Zero-Shot",
            "model": model_name,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "n_rows": int(len(data)),
            "n_features": int(X.shape[1]),
        })

    print(json.dumps({"results": results}, separators=(",", ":")))


if __name__ == "__main__":
    main()

{"results":[{"prompt_type":"Zero-Shot","model":"RandomForestRegressor","rmse":0.0010452801145144576,"mae":0.00011432038834954553,"r2":0.9938186464308872,"n_rows":2057,"n_features":10},{"prompt_type":"Zero-Shot","model":"GradientBoostingRegressor","rmse":0.0009519669606130155,"mae":0.0001246228338445003,"r2":0.9948730160196821,"n_rows":2057,"n_features":10}]}


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor


FILE_PATH = "Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv"
TARGET_COL = "Daily Max 8-hour Ozone Concentration (ppm)"
PRECIP_COL = "Daily Precipitation (mm)"
RANDOM_STATE = 42
TEST_SIZE = 0.30


def to_num(s: pd.Series) -> pd.Series:
    x = s.astype(str).str.strip()
    x = x.replace({"": np.nan, "None": np.nan, "nan": np.nan, "NaN": np.nan})
    x = x.str.replace(",", "", regex=False)
    return pd.to_numeric(x, errors="coerce")


def is_leakage_like(colname: str) -> bool:
    c = colname.lower()
    # anything ozone- or AQI-derived is suspicious for predicting ozone concentration
    return ("ozone" in c) or ("aqi" in c) or ("8-hour" in c) or ("8 hour" in c) or ("8hr" in c)


def is_date_like(colname: str) -> bool:
    c = colname.lower()
    return ("date" in c) or ("time" in c) or ("day" == c) or ("datetime" in c)


def is_meteo_like(colname: str) -> bool:
    c = colname.lower()
    meteo_keys = [
        "temp", "temperature",
        "dew", "dewpoint",
        "humid", "rh",
        "wind", "gust",
        "pressure", "baro",
        "precip", "rain", "snow",
        "cloud", "sky",
        "visibility",
        "sunrise", "sunset", "daylight",
        "solar", "radiation", "uv",
    ]
    return any(k in c for k in meteo_keys)


def build_meteorology_feature_list(df: pd.DataFrame) -> list:
    cols = [c for c in df.columns if c != TARGET_COL]
    # drop obvious non-feature columns
    cols = [c for c in cols if not is_date_like(c)]
    # take meteorology-like columns only
    cols = [c for c in cols if is_meteo_like(c)]
    # remove any leakage-like columns just in case they slipped in
    cols = [c for c in cols if not is_leakage_like(c)]
    # keep only columns that have some numeric signal
    keep = []
    for c in cols:
        non_missing = df[c].notna().sum()
        if non_missing >= max(30, int(0.30 * len(df))):
            keep.append(c)
    return keep


def median_impute(X: pd.DataFrame) -> pd.DataFrame:
    X2 = X.copy()
    for c in X2.columns:
        col = X2[c]
        if col.notna().any():
            X2[c] = col.fillna(col.median())
        else:
            X2[c] = col.fillna(0)
    return X2


def add_lags(dfX: pd.DataFrame, lag_cols: list, lags=(1, 2)) -> pd.DataFrame:
    out = dfX.copy()
    for lag in lags:
        shifted = dfX[lag_cols].shift(lag)
        shifted.columns = [f"{c}_lag{lag}" for c in lag_cols]
        out = pd.concat([out, shifted], axis=1)
    return out


def evaluate_rf(prompt_type: str, df: pd.DataFrame, base_features: list) -> dict:
    y = df[TARGET_COL].copy()

    if prompt_type == "Zero-Shot":
        X = df[base_features].copy()
        data = pd.concat([X, y], axis=1).dropna()
        X = data[base_features]
        y2 = data[TARGET_COL]

    elif prompt_type == "Few-Shot":
        X = df[base_features].copy()
        data = pd.concat([X, y], axis=1)
        data = data.dropna(subset=[TARGET_COL])
        X = data[base_features]
        y2 = data[TARGET_COL]
        X = median_impute(X)

    elif prompt_type == "Constraint-Based":
        X = df[base_features].copy()
        data = pd.concat([X, y], axis=1)
        data = data.dropna(subset=[TARGET_COL])
        X = data[base_features]
        y2 = data[TARGET_COL]
        X = median_impute(X)

    elif prompt_type == "Chain-of-Thought":
        X0 = df[base_features].copy()
        X1 = add_lags(X0, base_features, lags=(1, 2))
        data = pd.concat([X1, y], axis=1)
        data = data.dropna(subset=[TARGET_COL])
        X = data.drop(columns=[TARGET_COL])
        y2 = data[TARGET_COL]
        X = median_impute(X)

    else:
        raise ValueError(f"Unknown prompt_type: {prompt_type}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y2, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    model = RandomForestRegressor(random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    mae = float(mean_absolute_error(y_test, preds))
    r2 = float(r2_score(y_test, preds))

    return {
        "prompt_type": prompt_type,
        "model": "RandomForestRegressor",
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "n_rows": int(len(pd.concat([X, y2], axis=1))),
        "n_features": int(X.shape[1]),
        "features_used": list(X.columns),
    }


def main():
    df = pd.read_csv(FILE_PATH)
    df.columns = df.columns.astype(str).str.strip()

    if TARGET_COL not in df.columns:
        raise ValueError(f"Target column not found: {TARGET_COL}")

    if PRECIP_COL in df.columns:
        df[PRECIP_COL] = (
            df[PRECIP_COL].astype(str).str.strip().replace(
                {"T": "0", "t": "0", "Trace": "0", "trace": "0"}
            )
        )

    # numeric conversion (target + any meteo-ish candidates)
    df[TARGET_COL] = to_num(df[TARGET_COL])

    # convert everything that looks meteorological to numeric
    for c in df.columns:
        if c == TARGET_COL:
            continue
        if is_meteo_like(c) or (c == PRECIP_COL):
            df[c] = to_num(df[c])

    # show suspicious leakage-like columns (besides target)
    leak_cols = [c for c in df.columns if c != TARGET_COL and is_leakage_like(c)]
    if leak_cols:
        print("LEAKAGE_LIKE_COLUMNS=" + json.dumps(leak_cols, separators=(",", ":")))

    base_features = build_meteorology_feature_list(df)
    if not base_features:
        raise ValueError("No meteorology-like numeric features detected. Check your column names.")

    prompt_order = ["Zero-Shot", "Few-Shot", "Constraint-Based", "Chain-of-Thought"]

    results = []
    for p in prompt_order:
        results.append(evaluate_rf(p, df, base_features))

    # One JSON line
    print(json.dumps({"results": results}, separators=(",", ":")))


if __name__ == "__main__":
    main()

LEAKAGE_LIKE_COLUMNS=["AQI_Ozone"]
{"results":[{"prompt_type":"Zero-Shot","model":"RandomForestRegressor","rmse":0.0067956445879587945,"mae":0.0052866341463414625,"r2":0.713275928118482,"n_rows":2049,"n_features":9,"features_used":["Temp. (oC)","RH (%)","Pressure (mb)","Wind Speed (m/s)","Max. Temp. (oC)","Min. Temp.  (oC)","Daily Precipitation (mm)","Sunrise","Sunset"]},{"prompt_type":"Few-Shot","model":"RandomForestRegressor","rmse":0.006848658125719033,"mae":0.005170258899676375,"r2":0.7346438339715933,"n_rows":2057,"n_features":9,"features_used":["Temp. (oC)","RH (%)","Pressure (mb)","Wind Speed (m/s)","Max. Temp. (oC)","Min. Temp.  (oC)","Daily Precipitation (mm)","Sunrise","Sunset"]},{"prompt_type":"Constraint-Based","model":"RandomForestRegressor","rmse":0.006848658125719033,"mae":0.005170258899676375,"r2":0.7346438339715933,"n_rows":2057,"n_features":9,"features_used":["Temp. (oC)","RH (%)","Pressure (mb)","Wind Speed (m/s)","Max. Temp. (oC)","Min. Temp.  (oC)","Daily Precipita

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv")

target_col = "Daily Max 8-hour Ozone Concentration (ppm)"

if "Daily Precipitation (mm)" in df.columns:
    df["Daily Precipitation (mm)"] = df["Daily Precipitation (mm)"].replace("T", "0")
    df["Daily Precipitation (mm)"] = pd.to_numeric(df["Daily Precipitation (mm)"], errors="coerce")

met_keywords = ["temperature", "temp", "humidity", "rh", "relative hum", "pressure", "wind", "precipitation", "precip", "sunrise", "sunset", "dew"]
exclude_keywords = ["ozone", "aqi", "o3", "pollution", "co ", "no2", "so2", "pm2", "pm10"]

feature_cols = []
for col in df.columns:
    col_lower = col.lower()
    if col == target_col:
        continue
    if any(ex in col_lower for ex in exclude_keywords):
        continue
    if any(mk in col_lower for mk in met_keywords):
        feature_cols.append(col)

for col in feature_cols + [target_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[feature_cols + [target_col]].dropna()

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

result = {
    "prompt_type": "Zero-Shot",
    "model": "RandomForestRegressor",
    "rmse": round(rmse, 6),
    "mae": round(mae, 6),
    "r2": round(r2, 6),
    "n_rows": int(len(df)),
    "n_features": int(len(feature_cols))
}

print(json.dumps(result))

{"prompt_type": "Zero-Shot", "model": "RandomForestRegressor", "rmse": 0.006796, "mae": 0.005287, "r2": 0.713276, "n_rows": 2049, "n_features": 9}


In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1) Load CSV
df = pd.read_csv("Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv")

# 2) Clean numeric columns and handle non-numeric values safely
target_col = "Daily Max 8-hour Ozone Concentration (ppm)"

if "Daily Precipitation (mm)" in df.columns:
    df["Daily Precipitation (mm)"] = df["Daily Precipitation (mm)"].replace("T", "0")
    df["Daily Precipitation (mm)"] = pd.to_numeric(df["Daily Precipitation (mm)"], errors="coerce")

met_keywords = ["temperature", "temp", "humidity", "rh", "relative hum", "pressure", "wind", "precipitation", "precip", "sunrise", "sunset", "dew"]
exclude_keywords = ["ozone", "aqi", "o3", "pollution", "co ", "no2", "so2", "pm2", "pm10"]

feature_cols = []
for col in df.columns:
    col_lower = col.lower()
    if col == target_col:
        continue
    if any(ex in col_lower for ex in exclude_keywords):
        continue
    if any(mk in col_lower for mk in met_keywords):
        feature_cols.append(col)

for col in feature_cols + [target_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[feature_cols + [target_col]].dropna()

# 3) Define feature matrix X and target y
X = df[feature_cols]
y = df[target_col]

# 4) Perform train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# 5) Train models
models = {
    "RandomForestRegressor": RandomForestRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42)
}

# 6) Evaluate and 7) print results
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(mean_absolute_error(y_test, y_pred))
    r2 = float(r2_score(y_test, y_pred))

    result = {
        "prompt_type": "Few-Shot",
        "model": model_name,
        "rmse": round(rmse, 6),
        "mae": round(mae, 6),
        "r2": round(r2, 6),
        "n_rows": int(len(df)),
        "n_features": int(len(feature_cols))
    }

    print(json.dumps(result))

{"prompt_type": "Few-Shot", "model": "RandomForestRegressor", "rmse": 0.006796, "mae": 0.005287, "r2": 0.713276, "n_rows": 2049, "n_features": 9}
{"prompt_type": "Few-Shot", "model": "GradientBoostingRegressor", "rmse": 0.006928, "mae": 0.005408, "r2": 0.701972, "n_rows": 2049, "n_features": 9}


In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def load_and_clean(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    if "Daily Precipitation (mm)" in df.columns:
        df["Daily Precipitation (mm)"] = df["Daily Precipitation (mm)"].replace("T", "0")
        df["Daily Precipitation (mm)"] = pd.to_numeric(df["Daily Precipitation (mm)"], errors="coerce")
    return df


def select_features(df, target_col):
    met_keywords = ["temperature", "temp", "humidity", "rh", "relative hum",
                    "pressure", "wind", "precipitation", "precip", "sunrise", "sunset", "dew"]
    exclude_keywords = ["ozone", "aqi"]
    feature_cols = []
    for col in df.columns:
        col_lower = col.lower()
        if col == target_col:
            continue
        if any(ex in col_lower for ex in exclude_keywords):
            continue
        if any(mk in col_lower for mk in met_keywords):
            feature_cols.append(col)
    return feature_cols


def convert_to_numeric(df, cols):
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def add_lag_features(df, feature_cols):
    lag_cols = []
    for col in feature_cols:
        for lag in [1, 2]:
            lag_name = f"{col}_lag{lag}"
            df[lag_name] = df[col].shift(lag)
            lag_cols.append(lag_name)
    return df, lag_cols


def impute_medians(df, cols):
    for col in cols:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
    return df


def evaluate(y_test, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(mean_absolute_error(y_test, y_pred))
    r2 = float(r2_score(y_test, y_pred))
    return rmse, mae, r2


# --- Main workflow ---

target_col = "Daily Max 8-hour Ozone Concentration (ppm)"

df = load_and_clean("Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv")

feature_cols = select_features(df, target_col)

df = convert_to_numeric(df, feature_cols + [target_col])

df = df.dropna(subset=[target_col])

df, lag_cols = add_lag_features(df, feature_cols)

all_features = feature_cols + lag_cols

df = impute_medians(df, all_features)

X = df[all_features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse, mae, r2 = evaluate(y_test, y_pred)

result = {
    "prompt_type": "Chain-of-Thought",
    "model": "RandomForestRegressor",
    "rmse": round(rmse, 6),
    "mae": round(mae, 6),
    "r2": round(r2, 6),
    "n_rows": int(len(df)),
    "n_features": int(len(all_features))
}

print(json.dumps(result))

{"prompt_type": "Chain-of-Thought", "model": "RandomForestRegressor", "rmse": 0.006691, "mae": 0.005145, "r2": 0.746714, "n_rows": 2057, "n_features": 27}


In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


TARGET_COL = "Daily Max 8-hour Ozone Concentration (ppm)"
MET_KEYWORDS = ["temperature", "temp", "humidity", "rh", "relative hum",
                "pressure", "wind", "precipitation", "precip", "sunrise", "sunset", "dew"]
EXCLUDE_KEYWORDS = ["ozone", "aqi"]


def load_data(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    return df


def clean_precipitation(df):
    col = "Daily Precipitation (mm)"
    if col in df.columns:
        df[col] = df[col].replace("T", "0")
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def select_meteorological_features(df, target_col):
    features = []
    for col in df.columns:
        col_lower = col.lower()
        if col == target_col:
            continue
        if any(ex in col_lower for ex in EXCLUDE_KEYWORDS):
            continue
        if any(mk in col_lower for mk in MET_KEYWORDS):
            features.append(col)
    return features


def convert_to_numeric(df, cols):
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def impute_medians(df, cols):
    for col in cols:
        df[col] = df[col].fillna(df[col].median())
    return df


def train_and_evaluate(X_train, X_test, y_train, y_test):
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(mean_absolute_error(y_test, y_pred))
    r2 = float(r2_score(y_test, y_pred))
    return rmse, mae, r2


# --- Pipeline ---

df = load_data("Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv")

df = clean_precipitation(df)

feature_cols = select_meteorological_features(df, TARGET_COL)

df = convert_to_numeric(df, feature_cols + [TARGET_COL])

df = df.dropna(subset=[TARGET_COL])

df = impute_medians(df, feature_cols)

X = df[feature_cols]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

rmse, mae, r2 = train_and_evaluate(X_train, X_test, y_train, y_test)

result = {
    "prompt_type": "Role-Based",
    "model": "RandomForestRegressor",
    "rmse": round(rmse, 6),
    "mae": round(mae, 6),
    "r2": round(r2, 6),
    "n_rows": int(len(df)),
    "n_features": int(len(feature_cols))
}

print(json.dumps(result))

{"prompt_type": "Role-Based", "model": "RandomForestRegressor", "rmse": 0.006849, "mae": 0.00517, "r2": 0.734644, "n_rows": 2057, "n_features": 9}


In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

TARGET_COL = "Daily Max 8-hour Ozone Concentration (ppm)"

ALLOWED_FEATURES = [
    "Temp. (oC)",
    "RH (%)",
    "Pressure (mb)",
    "Wind Speed (m/s)",
    "Max. Temp. (oC)",
    "Min. Temp. (oC)",
    "Daily Precipitation (mm)",
    "Sunrise",
    "Sunset",
]

EXCLUDE_SUBSTRINGS = ["ozone", "aqi", "pm", "no2", "so2", "co"]


def load_data(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    return df


def validate_target(df):
    if TARGET_COL not in df.columns:
        raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")
    return df


def clean_precipitation(df):
    col = "Daily Precipitation (mm)"
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower().replace("t", "0")
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def select_features(df):
    present = [col for col in ALLOWED_FEATURES if col in df.columns]
    safe = []
    for col in present:
        col_lower = col.lower()
        if any(ex in col_lower for ex in EXCLUDE_SUBSTRINGS):
            continue
        safe.append(col)
    if not safe:
        raise ValueError("No allowed meteorological feature columns found in the dataset.")
    return safe


def convert_to_numeric(df, cols):
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def impute_medians(df, cols):
    for col in cols:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
    return df


# --- Pipeline ---

df = load_data("Daily_Weather_Polution_2017_2024_Ozone_Newark_Newark.csv")

df = validate_target(df)

df = clean_precipitation(df)

feature_cols = select_features(df)

df = convert_to_numeric(df, feature_cols + [TARGET_COL])

df = df.dropna(subset=[TARGET_COL])

df = impute_medians(df, feature_cols)

X = df[feature_cols]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

result = {
    "prompt_type": "Constraint-Based",
    "model": "RandomForestRegressor",
    "rmse": round(rmse, 6),
    "mae": round(mae, 6),
    "r2": round(r2, 6),
    "n_rows": int(len(df)),
    "n_features": int(len(feature_cols))
}

print(json.dumps(result))

{"prompt_type": "Constraint-Based", "model": "RandomForestRegressor", "rmse": 0.006865, "mae": 0.00519, "r2": 0.733367, "n_rows": 2057, "n_features": 8}
